# Design Document for `h_getBetaMix` from `phase1b`

**Author:** Audrey Yeo Te ying · [GitHub](https://github.com/audreyyeoCH) · [LinkedIn](https://linkedin.com/in/audrey-yeo-8000)
**Date:** October 2026
**Status:** Design document prototype only

# BetaMixturePrior: a Python translation of phase1b's `h_getBetamixPost`

This notebook re-implements the beta-mixture posterior update from the R package
[phase1b](https://github.com/Genentech/phase1b) as a Python class, to compare
functional and object-oriented designs.

**Design choices**

- **Validate once, at creation.** In phase1b, `par` and `weights` are passed
  separately between functions, so input checks are repeated in many places.
  Here, `__post_init__` validates the prior when it is created, and every
  function that receives a `BetaMixturePrior` can trust it.
- **Immutable objects.** `@dataclass(frozen=True)` means a prior cannot be
  changed after it is validated. `posterior()` returns a new object instead
  of modifying the existing one.
- **Strict rather than lenient.** phase1b renormalises weights that do not sum
  to 1 and issues a warning. This version raises an error instead, because in
  model code a silent correction is easier to miss than a failure.

The statistics are identical to the R version. Tests are written with `pytest`.

In [ ]:
import numpy as np
from dataclasses import dataclass
from scipy.special import betaln

For comparison the phase1b code for `h_getBetamixPost` is:
 ```r
h_getBetamixPost <- function(x, n, par, weights) {
  assert_number(x, lower = 0, upper = n, finite = TRUE)
  assert_number(n, lower = 0, finite = TRUE)
  assert_matrix(par, min.rows = 1, max.cols = 2, mode = "numeric")
  assert_numeric(
    weights,
    min.len = 0,
    len = nrow(par),
    finite = TRUE
  )
  assert_true(all(weights >= 0))
  # We now compute updated parameters.
  postPar <- par
  postPar[, 1] <- postPar[, 1] + x
  postPar[, 2] <- postPar[, 2] + n - x
  # We compute updated mixture probabilities.
  tmp <- exp(
    lbeta(a = postPar[, 1], b = postPar[, 2]) -
      lbeta(a = par[, 1], b = par[, 2])
  )
  # We compute the updated weights of the posterior
  postWeights <- weights * tmp / sum(weights * tmp)
  assert_numeric(postWeights)
  list(
    par = postPar,
    weights = postWeights
  )
}
```

In [ ]:

@dataclass(frozen=True)
class BetaMixturePrior:
    alpha: tuple[float, ...]
    beta: tuple[float, ...]
    weights: tuple[float, ...]

    def __post_init__(self):
        if not (len(self.alpha) == len(self.beta) == len(self.weights)):
            raise ValueError("alpha, beta and weights must have the same length")
        if any(a <= 0 or b <= 0 for a, b in zip(self.alpha, self.beta)):
            raise ValueError("alpha and beta must be positive")
        if any(w < 0 for w in self.weights):
            raise ValueError("weights must be non-negative")
        if not np.isclose(sum(self.weights), 1.0):
            raise ValueError("weights must sum to 1")

    def posterior(self, x: int, n: int) -> "BetaMixturePrior":
        alpha = np.array(self.alpha)
        beta = np.array(self.beta)
        updated_alpha = alpha + x              
        updated_beta = beta + (n - x)          
        log_w = np.log(self.weights) + betaln(updated_alpha, updated_beta) - betaln(alpha, beta)
        w = np.exp(log_w - np.max(log_w))  # subtract max for numerical stability
        w = w / np.sum(w)  # normalise to sum to 1
        return BetaMixturePrior(
        tuple(float(v) for v in updated_alpha),
        tuple(float(v) for v in updated_beta),
        tuple(float(v) for v in w),
        )

# Example usage :
prior = BetaMixturePrior(alpha=(2, 3), beta=(5, 4), weights=(0.6, 0.4))
post = prior.posterior(x = 6, n = 10)   
print(post)

BetaMixturePrior(alpha=(8.0, 9.0), beta=(9.0, 8.0), weights=(0.4285714285714288, 0.5714285714285712))


In [ ]:
import pytest

def test_weights_must_sum_to_one():
    with pytest.raises(ValueError, match="sum to 1"):
        BetaMixturePrior(alpha=(2, 3), beta=(5, 4), weights=(0.6, 1.4))
    with pytest.raises(ValueError, match="n < x"):
        prior.posterior(x = 11, n = 10)
    with pytest.raises(ValueError, match="alpha and beta must be positive"):
        BetaMixturePrior(alpha=(-2, 3), beta=(5, 4), weights=(0.6, 0.4))